# MR Optimum Mode 2 — AWS Setup and Teardown

This notebook deploys an MR Optimum Mode 2 worker into your AWS account and registers it with your existing CloudMRHub account.

> **Security:** Use a dedicated IAM user approved for Mode 2 deployment. Secrets are requested through hidden prompts and are not stored in the notebook. The AWS profile is written only to the temporary Colab runtime. Never paste credentials into a text or code cell.

## Before running the notebook

You need:

1. A working CloudMRHub email and password.
2. A dedicated AWS IAM user.
3. An Access Key ID and Secret Access Key created under **IAM → Users → your user → Security credentials → Access keys**.
4. IAM permission to deploy CloudFormation, IAM roles/pass-role, Lambda, API Gateway, ECS/Fargate, EC2 security groups, S3, CloudWatch Logs, EventBridge, and STS identity checks.
5. A VPC subnet with outbound internet access.

Console username/password and CLI access keys are separate. Do not use AWS root credentials.

In [ ]:
# Step 1 — Download MR Optimum and install the manager dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/content/mroptimum-app')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/cloudmrhub/mroptimum-app.git', str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boto3', 'requests'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'Ready: {REPO_DIR}')

## Configure the temporary AWS profile

The next cell asks for the Access Key ID and Secret Access Key using hidden inputs. It creates the profile named **mroptimum** inside this Colab runtime. Values entered through getpass are not displayed or saved in the notebook.

In [ ]:
# Step 2 — Create the AWS profile without displaying secrets
import configparser
from getpass import getpass

AWS_PROFILE = 'mroptimum'
AWS_REGION = 'us-east-1'

access_key_id = getpass('AWS Access Key ID: ')
secret_access_key = getpass('AWS Secret Access Key: ')
if not access_key_id or not secret_access_key:
    raise ValueError('Both AWS credentials are required.')

aws_dir = Path.home() / '.aws'
aws_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
credentials_path = aws_dir / 'credentials'
config_path = aws_dir / 'config'

credentials = configparser.RawConfigParser()
if credentials_path.exists():
    credentials.read(credentials_path)
if not credentials.has_section(AWS_PROFILE):
    credentials.add_section(AWS_PROFILE)
credentials.set(AWS_PROFILE, 'aws_access_key_id', access_key_id)
credentials.set(AWS_PROFILE, 'aws_secret_access_key', secret_access_key)
with credentials_path.open('w', encoding='utf-8') as handle:
    credentials.write(handle)
os.chmod(credentials_path, 0o600)

configuration = configparser.RawConfigParser()
if config_path.exists():
    configuration.read(config_path)
profile_section = f'profile {AWS_PROFILE}'
if not configuration.has_section(profile_section):
    configuration.add_section(profile_section)
configuration.set(profile_section, 'region', AWS_REGION)
configuration.set(profile_section, 'output', 'json')
with config_path.open('w', encoding='utf-8') as handle:
    configuration.write(handle)
os.chmod(config_path, 0o600)

del access_key_id, secret_access_key
print(f'AWS profile created: {AWS_PROFILE} ({AWS_REGION})')

In [ ]:
# Step 3 — Verify the AWS account before deploying
import boto3

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
identity = session.client('sts').get_caller_identity()
print('AWS account:', identity['Account'])
print('IAM identity:', identity['Arn'])
print('Confirm that this is the intended account before continuing.')

## Deploy Mode 2

Enter the existing CloudMRHub email, a recognizable worker alias, and the CloudMRHub password. The password is hidden and is passed to the manager in memory. Do **not** enter an AWS console password or Secret Access Key.

In [ ]:
# Step 4 — Deploy and register the Mode 2 worker
import argparse
from getpass import getpass
from worker import manage as mode2

cloudmr_email = input('CloudMRHub email: ').strip()
worker_alias = input('Worker alias [My Mode 2 Worker]: ').strip() or 'My Mode 2 Worker'
cloudmr_password = getpass('CloudMRHub password: ')

deploy_args = argparse.Namespace(
    profile=AWS_PROFILE, region=AWS_REGION, email=cloudmr_email,
    password=cloudmr_password, api_key=None, alias=worker_alias,
    image_uri=None, yes=False, follow=False, minutes=30, command='deploy'
)
try:
    mode2.cmd_deploy(deploy_args)
finally:
    deploy_args.password = None
    del cloudmr_password

In [ ]:
# Step 5 — Check the deployed worker
status_args = argparse.Namespace(profile=AWS_PROFILE, region=AWS_REGION)
mode2.cmd_status(status_args)

A successful deployment reports **CREATE_COMPLETE**, an API Gateway endpoint, and **Health: OK**. Sign in to MR Optimum with the same CloudMRHub account, refresh the computing-unit list, and select the new worker alias.

In [ ]:
# Optional — Update the worker to the current template and image
run_update = input('Type UPDATE to update the worker, or press Enter to skip: ').strip()
if run_update == 'UPDATE':
    update_args = argparse.Namespace(
        profile=AWS_PROFILE, region=AWS_REGION, image_uri=None
    )
    mode2.cmd_update(update_args)
else:
    print('Update skipped.')

## Tear down Mode 2

Run the next cell when the worker is no longer needed. It deregisters the computing unit from CloudMRHub, clears temporary payload objects, deletes the CloudFormation stack, and removes the local worker configuration.

> Keep the Colab runtime connected until deletion finishes. If the runtime is lost, rerun the setup and AWS-profile cells before running teardown.

In [ ]:
# Destructive — Deregister and delete the Mode 2 worker
from getpass import getpass

confirmation = input('Type DELETE to remove the Mode 2 worker: ').strip()
if confirmation == 'DELETE':
    teardown_email = input('CloudMRHub email: ').strip()
    teardown_password = getpass('CloudMRHub password: ')
    teardown_args = argparse.Namespace(
        profile=AWS_PROFILE, region=AWS_REGION, email=teardown_email,
        password=teardown_password, yes=True
    )
    try:
        mode2.cmd_teardown(teardown_args)
    finally:
        teardown_args.password = None
        del teardown_password
else:
    print('Teardown cancelled.')

## Final security cleanup

After a successful teardown:

- Refresh MR Optimum and confirm that the worker alias is gone.
- In Colab, use **Runtime → Disconnect and delete runtime** to remove the temporary credential files.
- Rotate or delete the IAM access key when it is no longer required.
- If teardown reports a deregistration warning, the AWS stack may be gone while a stale worker entry still requires manual removal.